# SynopsisLake geometric histogram output review

This notebook reads the synopsis Parquet written by the `SynopsisLake - CI JAR Run` job (`SynopsisLake.SynopsisOverheadExp`) and explains what the output means, linked to the Java source and the paper *SynopsisLake: Quality-aware Approximate Spatial Query Processing Using Data Synopses* (Zhang & Eldawy, SIGSPATIAL '25, [doi 10.1145/3748636.3762714](https://dl.acm.org/doi/10.1145/3748636.3762714)).

**Run being reviewed**

| Setting | Value |
| --- | --- |
| Input | `/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join/0.csv` |
| `-synopsisType` | `4` (Geometric Histogram) |
| `-synopsisResolution` | `128` (128 x 128 grid per synopsis) |
| `-numFile` / `-numOfRecordPerFile` | `5` / `10000` (each chunk = 1 id slot + 9,999 records) |
| `-writeDataFileInt` | `0`, which means **write** the `.bin` files (the flag is inverted in `main`) |
| Output | `.../synopsis_lake/output/synout/file-<id>.bin` (raw boxes) and `.../synout/syno/` (synopsis Parquet) |
| Runtime | Job cluster DBR 14.3 LTS (Spark 3.5.0, Scala 2.12), single user; JAR built against Spark 3.0.1 |

**Cell guide**
* Cells 2-4: sanity checks and the raw Parquet read.
* Cell 5: what each Parquet column means and which Java code produced it.
* Cells 6-11: walkthrough of the geometric histogram (data profile, decoded histogram, query estimate), each followed by a findings note.
* Last cell: how the engine uses this, findings to discuss, and Databricks porting notes.

In [0]:
import os
print(os.getcwd())

In [0]:
parquet_path = "/Workspace/Shared/SynopsisLake/experiments/OutputWriteFileDataInt.parquet"
df = spark.read.parquet(parquet_path)
display(df.limit(100))

# Basic summary
print("Count:", df.count())
df.describe().display()

In [0]:
# Read parquet output from UC Volume (written by SynopsisOverheadExp JAR)
parquet_path = "/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/output/synout/syno/"
df = spark.read.parquet(parquet_path)
display(df.limit(100))

# Basic summary
print("Count:", df.count())
df.describe().display()

## What the synopsis Parquet contains

Each row is one synopsis for one chunk of the input. This is the *synopsis Parquet file* from paper section 2.1: **information columns** describe the summarized data and the partitioning, and **content columns** hold the statistics.

| Column | Group | Source in Java | Meaning |
| --- | --- | --- | --- |
| `id` | info | chunk id `t.get(0)` | 0-4, one per chunk |
| `type` | info | `synopsisType` | Always 4 (geometric histogram) |
| `totalNumOfItems` | info | `t.size() - 1` | 9,999 records per chunk |
| `rawFilePath` | info | `HDFSURI/synout/file-<id>.bin` | Raw data file this synopsis summarizes |
| `range` | info | `DistributedBinaryData.buildFile` | Chunk bounding box, order **`[MinLon, MaxLon, MinLat, MaxLat]`**. Max values include the +0.35 / +0.17 box padding. |
| `shapes` | info | `getLonUnit()`, `getLatUnit()` | Grid cell size = range width / 128 (the partition function) |
| `corners` | content | `GeometricHistogramOpt.count` | Rectangle corner points falling in each cell |
| `area` | content | `mSumRatioArea` | Sum of (overlap area / cell area) over rectangles |
| `horizontal` | content | `mSumRatioH` | Horizontal edge length inside the cell / cell width |
| `vertical` | content | `mSumRatioV` | Vertical edge length inside the cell / cell height |

All four grids are indexed `[lonBucket][latBucket]`.

**Code path that produced each row:** `loadData` (driver) splits the CSV into chunks, then `lineRDD.foreach(WriteBinaryFileToHDFS)` runs one task per chunk on the workers. `DistributedBinaryData.buildFile` writes `file-<id>.bin` and builds `GeometricHistogramOpt` (`addRecord` per rectangle). The row comes back through `BinaryFileSizeAccumulator` and `writeParquetFile` writes it with Spark.

**Note on the input boxes:** `buildFile` uses only CSV columns 0-1 (`minLon`, `minLat`) and builds a synthetic box `[lon, lat, lon+0.35, lat+0.17]` (code comment: *"make a two-dimensional polygon dataset"*). The real MBR columns 2-3 are ignored.

## Geometric histogram walkthrough on spatial_join/0.csv

The job split `0.csv` into 5 chunks of 9,999 rows (rows `k*9999 .. (k+1)*9999-1` form chunk `k`). For each chunk it built a 128x128 geometric histogram over boxes `[lon, lat, lon+0.35, lat+0.17]` built from columns 0-1 only. The cells below:
1. Profile the real rectangles in the CSV and map the chunks.
2. Decode one histogram row into geographic heatmaps.
3. Use the histogram the way the engine does: estimate how many objects intersect a query window, then compare the estimate with the exact count.

**Where this sits in the paper**
* Table 1: the Geometric Histogram (An et al.) is the synopsis SynopsisLake uses for `SpatialJoinEst` (spatial join selectivity estimation).
* Section 5: polygon datasets (OSM15/lakes, OSM15/parks) are used to test geometric histograms. `0.csv` contains polygon MBRs, which matches that setup. Which OSM layer `0.csv` comes from is not documented in the repo.
* Section 5, Table 3 measures the overhead of creating 50 synopsis files; `SynopsisOverheadExp` defaults to `numFile = 50`, so this class is that experiment harness. Our CI run is a 5-file version.
* Table 4 reports 1.281 ms join estimation latency at 128 x 128 resolution, the resolution used here.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_path = "/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/data/spatial_join/0.csv"
NUM_FILE, PER_FILE = 5, 9999          # NUM_RECORDS=10000 minus the chunk-id slot
LON_PAD, LAT_PAD = 0.35, 0.17         # synthetic box size used in DistributedBinaryData

raw = pd.read_csv(csv_path, sep=";", header=None,
                  names=["minLon", "minLat", "maxLon", "maxLat", "oid", "c5", "c6"])
raw["chunk"] = np.arange(len(raw)) // PER_FILE
used = raw[raw.chunk < NUM_FILE].copy()

# Real MBR size vs the synthetic box the Java code actually histogrammed
used["w_deg"] = used.maxLon - used.minLon
used["h_deg"] = used.maxLat - used.minLat
print(f"Rows in CSV: {len(raw):,}; rows used by the job: {len(used):,}")
display(used[["w_deg", "h_deg"]].describe(percentiles=[.5, .9, .99]).T
        .assign(synthetic_box=[LON_PAD, LAT_PAD]))

# Per-chunk bounding box as the job computes it (min of lon/lat, max of lon+0.35 / lat+0.17)
chunk_rng = used.groupby("chunk").agg(minLon=("minLon", "min"), maxLon=("minLon", lambda s: s.max() + LON_PAD),
                                      minLat=("minLat", "min"), maxLat=("minLat", lambda s: s.max() + LAT_PAD),
                                      n=("oid", "size"))
display(chunk_rng)

fig, ax = plt.subplots(figsize=(7, 7))
for k, g in used.groupby("chunk"):
    ax.scatter(g.minLon, g.minLat, s=1, alpha=0.4, label=f"chunk {k}")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_aspect("equal")
ax.set_title("Rectangle lower-left corners in 0.csv (first 5 x 9,999 rows)")
ax.legend(markerscale=10)
plt.show()

### What the profile shows

* `0.csv` has 97,750 rows: `minLon; minLat; maxLon; maxLat; oid; c5; c6`. Each row is one object's bounding box (MBR). `c5` takes 0/1/2 and `c6` is always 1; their meaning is not documented.
* The job uses only the first 49,995 rows (5 chunks x 9,999).
* Real objects are tiny: the median MBR is about 0.002 x 0.0018 degrees (roughly 200 m). The synthetic box the code builds (0.35 x 0.17 degrees) is over 100 times larger.
* The per-chunk bounding boxes computed here match the Parquet `range` column exactly (for example chunk 3: `[-178.85, -35.69, -55.98, 18.17]`), which confirms the chunking and the column order.
* Most objects lie in South America (roughly lon -80 to -35, lat -56 to 18). A few outliers near lon -179 stretch every chunk's grid, so cells are about 1.1 x 0.6 degrees and most data falls into a few columns.

In [0]:
# Decode the geometric histogram for one chunk; arrays are [lonBucket][latBucket]
CHUNK = 0
syn = spark.read.parquet("/Volumes/cs598_ccc_research_7474657373242960/synopsis_lake/output/synout/syno/")
row = syn.filter(f"id = {CHUNK}").first()

min_lon, max_lon, min_lat, max_lat = row["range"]      # order: [MinLon, MaxLon, MinLat, MaxLat]
lon_unit, lat_unit = row["shapes"]
grids = {k: np.array(row[k]) for k in ["corners", "area", "horizontal", "vertical"]}
print(f"chunk {CHUNK}: lon [{min_lon:.3f}, {max_lon:.3f}], lat [{min_lat:.3f}, {max_lat:.3f}], "
      f"cell = {lon_unit:.4f} deg x {lat_unit:.4f} deg, items = {row['totalNumOfItems']}")
print("sum(corners) =", grids["corners"].sum(), "(about 4 per box)")
print("sum(area) =", round(grids["area"].sum(), 2),
      "(about items x box area / cell area =",
      round(row["totalNumOfItems"] * LON_PAD * LAT_PAD / (lon_unit * lat_unit), 2), ")")

extent = [min_lon, max_lon, min_lat, max_lat]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, g) in zip(axes, grids.items()):
    im = ax.imshow(np.log1p(g.T), origin="lower", extent=extent, aspect="equal", cmap="viridis")
    ax.set_title(f"{name} (log1p)"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout(); plt.show()

### What the decoded histogram shows (chunk 0)

* `sum(area)` = 854.2, exactly 9,999 x (0.35 x 0.17) / cell area, so `area` is total box coverage in units of cell area.
* `sum(corners)` = 39,998 = 4 corners x 9,999 boxes + 2 extra from a zero-size phantom record (see below).
* **Phantom record:** the loops in `buildFile` start at `i = 1`, so `dataArray[0]` stays `{0,0,0,0}` but is still passed to `addRecord`. `getBId` clamps it into the last longitude bucket, which is also the source of the `log1p` warning (negative values in that bucket). The next cell skips that bucket.
* **Each chunk has its own grid:** chunk 0 cells are 1.125 x 0.619 degrees, chunk 3 cells are 1.1185 x 0.579, and chunk 4 starts at lat -89.97. The same `[i][j]` therefore covers different ground in different rows, so rows cannot be added cell by cell. This is the "unmergeable synopses" problem from paper section 1.

In [0]:
# Estimate how many objects intersect a query window using the geometric histogram
# (same formula as GeometricHistogramOpt.selectivityEstimation), then compare with exact counts.

def gh_for_rect(qminLon, qminLat, qmaxLon, qmaxLat, nlon, nlat):
    """Geometric histogram of a single rectangle on the chunk grid: corners, area ratio, H, V."""
    C, A, H, V = (np.zeros((nlon, nlat)) for _ in range(4))
    lon_edges = min_lon + np.arange(nlon + 1) * lon_unit
    lat_edges = min_lat + np.arange(nlat + 1) * lat_unit
    for i in range(nlon):
        cx0, cx1 = lon_edges[i], lon_edges[i + 1]
        ix0, ix1 = max(qminLon, cx0), min(qmaxLon, cx1)
        if ix1 <= ix0:
            continue
        for j in range(nlat):
            cy0, cy1 = lat_edges[j], lat_edges[j + 1]
            iy0, iy1 = max(qminLat, cy0), min(qmaxLat, cy1)
            if iy1 <= iy0:
                continue
            A[i, j] = (ix1 - ix0) * (iy1 - iy0) / (lon_unit * lat_unit)
            C[i, j] = sum(cx0 <= x < cx1 and cy0 <= y < cy1
                          for x in (qminLon, qmaxLon) for y in (qminLat, qmaxLat))
            H[i, j] = (ix1 - ix0) / lon_unit * sum(cy0 <= y < cy1 for y in (qminLat, qmaxLat))
            V[i, j] = (iy1 - iy0) / lat_unit * sum(cx0 <= x < cx1 for x in (qminLon, qmaxLon))
    return C, A, H, V

# Query window: a 1 deg x 1 deg box centred on the densest histogram cell of this chunk
i, j = np.unravel_index(np.argmax(grids["corners"][:-1, :]), grids["corners"][:-1, :].shape)  # skip last lon bucket (phantom record)
cx, cy = min_lon + (i + 0.5) * lon_unit, min_lat + (j + 0.5) * lat_unit
q = (cx - 0.5, cy - 0.5, cx + 0.5, cy + 0.5)
print(f"Query window lon [{q[0]:.3f}, {q[2]:.3f}], lat [{q[1]:.3f}, {q[3]:.3f}]")

nlon, nlat = grids["corners"].shape
Cq, Aq, Hq, Vq = gh_for_rect(*q, nlon, nlat)
est = np.ceil((grids["corners"] * Aq + grids["area"] * Cq
               + grids["horizontal"] * Vq + grids["vertical"] * Hq).sum() / 4)

chunk_df = used[used.chunk == CHUNK]
def intersects(x0, y0, x1, y1):
    return (x0 < q[2]) & (x1 > q[0]) & (y0 < q[3]) & (y1 > q[1])
true_synth = intersects(chunk_df.minLon, chunk_df.minLat, chunk_df.minLon + LON_PAD, chunk_df.minLat + LAT_PAD).sum()
true_real = intersects(chunk_df.minLon, chunk_df.minLat, chunk_df.maxLon, chunk_df.maxLat).sum()

display(pd.DataFrame({
    "measure": ["GH estimate (histogram only)", "exact, synthetic 0.35x0.17 boxes", "exact, real MBRs from CSV"],
    "objects_intersecting_query": [int(est), int(true_synth), int(true_real)],
}))

### What the query estimate shows

The query window is a 1 x 1 degree box around chunk 0's densest cell (lon -55.86 to -54.86, lat -27.83 to -26.83, Misiones province, Argentina). It is turned into a one-rectangle histogram on the same grid and combined with the chunk histogram using the formula in `GeometricHistogramOpt.selectivityEstimation()`: sum over cells of (C x A' + A x C' + H x V' + V x H') / 4, where C, A, H, V are corners, area, horizontal, vertical.

| Measure | Objects hitting the window |
| --- | --- |
| Estimate from the histogram only | 538 |
| Exact, synthetic 0.35 x 0.17 boxes | 590 |
| Exact, real MBRs from the CSV | 432 |

The estimate is within about 9% of the exact count for the boxes it was built from, without reading any raw rows. The gap to the real MBRs comes from the synthetic boxes, not from the histogram.

## How the engine uses this (paper sections 2-4)

Example: estimate how many objects in `0.csv` intersect objects in `1.csv` before running the join.
1. **Ingest:** each data file gets its own synopsis at load time (what this job did: `.bin` + one Parquet row per chunk).
2. **Find candidates:** the synopsis index and each row's `range` select the synopses that overlap the query.
3. **Align-Reshape-Merge (section 4):** align chunks to shared boundaries, reshape each histogram onto them, and merge. Geometric histogram statistics are ratio-type, so they are reshaped by the inverse ratio (source range / reshaped range); count-type values scale by overlap.
4. **Estimate:** apply the formula above to the merged `0.csv` and `1.csv` histograms to get the number of intersecting pairs. Section 3's Skewness-Align metric (bucket density, MAD / median, beta-distribution error model) attaches a quality score. The optimizer uses both to plan the join before opening any `.bin` file.

The previous cell is the single-window version of step 4, on one chunk with no merging.

## Findings to discuss

* **Harness ignores real geometry:** `DistributedBinaryData` reads only `minLon`, `minLat` and builds 0.35 x 0.17 boxes. The paper tests geometric histograms on real polygons, so to match it we should build boxes from all four CSV columns.
* **Phantom record at index 0** in every histogram (loops start at `i = 1` but `dataArray[0]` is still added).
* **Inverted flag:** `-writeDataFileInt 0` means *write* the `.bin` files.
* **Delimiter:** `regex` was hard-coded to tab; the CSV uses `;`. Fixed in `SynopsisOverheadExp.java` line 33. A `-regex` job parameter would avoid this per dataset.
* **Open question:** which dataset `0.csv` comes from, and what `c5` / `c6` mean.

## Databricks porting notes

* Use the shared `SparkSession.builder().getOrCreate()`; do not create a `JavaSparkContext` or call `sc.close()`.
* Startup Spark settings (executor memory, off-heap, `rpc.message.maxSize`) live in the job cluster `spark_conf` in `build-upload-and-run.yml`. CI updates the job from that file on every run.
* Java file I/O on `/Volumes` needs DBR 14.3 LTS or above on single-user compute (13.3 fails with `Operation not permitted`). `/Workspace` files are not readable from `java.io` in a JAR task, so inputs go in the `data` volume.
* Volumes: `jars` (JAR only), `data` (input CSVs uploaded by CI), `output` (results). Spark writes use `/Volumes/...` without the `file:` prefix.